# 高级 EDA：基于游戏行为数据的学生表现分析

本 Notebook 基于 Kaggle 竞赛 **Predict Student Performance from Game Play** 数据，完成课程项目所需的数据整理、清洗、描述性统计、探索性分析和可视化展示。

本版重点不是复杂建模，而是在原有分析基础上加入三个更有过程解释力的高级 EDA 维度：

1. **时间维度**：将每个游戏会话按事件顺序标准化为五个进程阶段，比较高低表现学生的行为节奏。
2. **行为路径维度**：构建事件类型转移矩阵，比较高低表现学生的操作路径差异。
3. **空间维度**：基于屏幕坐标绘制高低表现学生的交互密度图，比较空间探索模式。

In [1]:
from pathlib import Path
import os
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind, f_oneway, pearsonr
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_SECTION
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

warnings.filterwarnings("ignore")

ROOT = Path(r"D:\课程\数据分析")
DATA_DIR = ROOT / "predict-student-performance-from-game-play"
FIG_DIR = ROOT / "report_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 1_500_000

plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 180
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", palette="Set2", font="Microsoft YaHei")

def save_fig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return str(path)

def q_group(q):
    return "0-4" if q <= 3 else ("5-12" if q <= 13 else "13-22")

print("ROOT:", ROOT)
print("FIG_DIR:", FIG_DIR)

ROOT: D:\课程\数据分析
FIG_DIR: D:\课程\数据分析\report_figures


## 1. 数据读取、类型优化与标签解析

In [2]:
dtype_map = {
    "session_id": "int64", "index": "int32", "elapsed_time": "int64",
    "event_name": "category", "name": "category", "level": "int8",
    "room_coor_x": "float32", "room_coor_y": "float32",
    "screen_coor_x": "float32", "screen_coor_y": "float32",
    "hover_duration": "float32", "text": "category", "fqid": "category",
    "room_fqid": "category", "text_fqid": "category",
    "fullscreen": "int8", "hq": "int8", "music": "int8",
    "level_group": "category",
}

tdf = pd.read_csv(DATA_DIR / "train.csv", nrows=SAMPLE_SIZE, dtype=dtype_map)
tl = pd.read_csv(DATA_DIR / "train_labels.csv")

last_sid = tdf["session_id"].iloc[-1]
n_trunc = int((tdf["session_id"] == last_sid).sum())
tdf = tdf.loc[tdf["session_id"] != last_sid].copy()

labels = tl.copy()
labels[["session", "question"]] = labels["session_id"].str.extract(r"(\d+)_q(\d+)")
labels["session"] = labels["session"].astype("int64")
labels["question"] = labels["question"].astype(int)
labels["q_group"] = labels["question"].map(q_group)

print(f"读取日志行数: {len(tdf):,}，剔除边界截断会话 {last_sid} 的 {n_trunc:,} 行")
print(f"抽样日志会话数: {tdf['session_id'].nunique():,}")
print(f"标签记录数: {len(labels):,}，标签会话数: {labels['session'].nunique():,}")
print(f"抽样日志内存占用: {tdf.memory_usage(deep=True).sum()/1e6:.1f} MB")

读取日志行数: 1,499,566，剔除边界截断会话 20110014410950960 的 434 行
抽样日志会话数: 1,337
标签记录数: 424,116，标签会话数: 23,562
抽样日志内存占用: 103.6 MB


## 2. 会话级特征工程

In [3]:
def extract_session_features(df):
    base = df.groupby("session_id").agg(
        total_events=("index", "count"),
        total_elapsed_time=("elapsed_time", "max"),
        mean_elapsed_time=("elapsed_time", "mean"),
        std_elapsed_time=("elapsed_time", "std"),
        unique_levels=("level", "nunique"),
        max_level=("level", "max"),
        mean_hover=("hover_duration", "mean"),
        total_hover=("hover_duration", "sum"),
    ).reset_index()

    event_counts = df.groupby(["session_id", "event_name"], observed=True).size().unstack(fill_value=0)
    event_pct = event_counts.div(event_counts.sum(axis=1), axis=0) * 100
    event_pct.columns = [f"event_pct_{c}" for c in event_pct.columns]
    event_pct = event_pct.reset_index()

    settings = df.groupby("session_id").agg(
        fullscreen_mode=("fullscreen", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
        hq_mode=("hq", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
        music_mode=("music", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
    ).reset_index()

    out = base.merge(event_pct, on="session_id", how="left").merge(settings, on="session_id", how="left")
    out["time_per_event"] = out["total_elapsed_time"] / out["total_events"].clip(lower=1)
    return out.replace([np.inf, -np.inf], np.nan).fillna(0)

features = extract_session_features(tdf)
session_perf = labels.groupby("session")["correct"].agg(["mean", "count", "sum"]).reset_index()
session_perf.columns = ["session_id", "avg_correct", "questions_answered", "questions_correct"]
session_perf["avg_correct"] = session_perf["avg_correct"] * 100

adf = features.merge(session_perf, on="session_id", how="inner")
adf["performance_group"] = pd.cut(
    adf["avg_correct"],
    bins=[-0.1, 40, 60, 80, 100.1],
    labels=["低表现(0-40%)", "中等偏下(40-60%)", "中等偏上(60-80%)", "高表现(80-100%)"],
)
median_correct = adf["avg_correct"].median()
adf["high_performer"] = (adf["avg_correct"] >= median_correct).astype(int)
adf["perform_label"] = np.where(adf["high_performer"] == 1, "高表现组", "低表现组")

print(f"最终行为分析样本: {len(adf):,} 个会话，{adf.shape[1]} 个变量")
print(adf[["avg_correct", "total_events", "total_elapsed_time", "time_per_event"]].describe().round(2))

最终行为分析样本: 1,337 个会话，30 个变量
       avg_correct  total_events  total_elapsed_time  time_per_event
count      1337.00       1337.00        1.337000e+03         1337.00
mean         69.60       1121.59        5.413535e+06         5161.16
std          17.97        361.87        2.040734e+07        21906.20
min          11.11        645.00        5.633500e+05          357.01
25%          61.11        896.00        1.562362e+06         1552.82
50%          72.22       1017.00        2.102277e+06         1953.47
75%          83.33       1226.00        3.021359e+06         2638.43
max         100.00       4174.00        4.293728e+08       577113.96


## 3. 描述性统计与原有结论修正

In [4]:
question_stats = labels.groupby("question")["correct"].agg(["mean", "count"]).reset_index()
question_stats["mean_pct"] = question_stats["mean"] * 100
question_sorted = question_stats.sort_values("mean_pct")

level_group_stats = labels.groupby("q_group", sort=False)["correct"].agg(["mean", "count"])
level_group_stats["mean_pct"] = level_group_stats["mean"] * 100
group_arrays = [labels.loc[labels["q_group"] == g, "correct"].values for g in ["0-4", "5-12", "13-22"]]
level_f, level_p = f_oneway(*group_arrays)

overall_correct = labels["correct"].mean() * 100
mean_correct = adf["avg_correct"].mean()
median_correct = adf["avg_correct"].median()
std_correct = adf["avg_correct"].std()
event_corr, event_p = pearsonr(adf["total_events"], adf["avg_correct"])
time_corr, time_p = pearsonr(adf["total_elapsed_time"], adf["avg_correct"])

print("题目官方等级组正确率")
print(level_group_stats[["mean_pct", "count"]].round(2))
print(f"ANOVA: F={level_f:.2f}, p={level_p:.4g}")
print("最难题目")
print(question_sorted.head(5)[["question", "mean_pct"]].round(1).to_string(index=False))
print("最易题目")
print(question_sorted.tail(5)[["question", "mean_pct"]].round(1).to_string(index=False))
print(f"耗时与正确率相关: r={time_corr:.3f}, p={time_p:.4g}")
print(f"事件数与正确率相关: r={event_corr:.3f}, p={event_p:.4g}")

题目官方等级组正确率
         mean_pct   count
q_group                  
0-4         88.01   70686
5-12        64.99  235620
13-22       71.24  117810
ANOVA: F=7189.15, p=0
最难题目
 question  mean_pct
       13      27.5
       15      48.1
       10      50.5
        5      54.8
        8      61.7
最易题目
 question  mean_pct
        4      79.8
       12      86.3
        3      93.4
       18      95.1
        2      97.9
耗时与正确率相关: r=0.013, p=0.6279
事件数与正确率相关: r=-0.547, p=5.432e-105


## 4. 基础 EDA 图表

In [5]:
fig_paths = {}

# 图1
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
vc = labels["correct"].value_counts().sort_index()
ax[0].pie(vc.values, labels=["错误", "正确"], autopct="%1.1f%%", colors=["#F26D6D", "#4DB6AC"], startangle=90)
ax[0].set_title("总体答题正确率", fontweight="bold")
ax[1].hist(adf["avg_correct"], bins=30, color="#4C78A8", edgecolor="white", alpha=0.9)
ax[1].axvline(mean_correct, color="#D62728", ls="--", lw=1.5, label=f"均值 {mean_correct:.1f}%")
ax[1].axvline(median_correct, color="#FF7F0E", ls="--", lw=1.5, label=f"中位数 {median_correct:.1f}%")
ax[1].set_xlabel("会话平均正确率(%)")
ax[1].set_ylabel("会话数")
ax[1].set_title("会话正确率分布", fontweight="bold")
ax[1].legend(fontsize=8)
fig.tight_layout()
fig_paths["fig1"] = save_fig(fig, "fig01.png")

# 图2
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
qss = question_sorted.copy()
colors = plt.cm.RdYlGn(qss["mean_pct"].values / 100)
bars = axes[0].bar(range(len(qss)), qss["mean_pct"], color=colors, edgecolor="white")
axes[0].set_xticks(range(len(qss)))
axes[0].set_xticklabels([f"Q{q}" for q in qss["question"]], fontsize=8)
axes[0].set_ylabel("正确率(%)")
axes[0].set_title("各题目正确率排序", fontweight="bold")
axes[0].axhline(qss["mean_pct"].mean(), color="red", ls="--", lw=1)
for b, v in zip(bars, qss["mean_pct"]):
    axes[0].text(b.get_x() + b.get_width()/2, v + 0.6, f"{v:.0f}", ha="center", fontsize=6)
qo = qss["question"].tolist()
axes[1].boxplot([labels.loc[labels["question"] == q, "correct"].values for q in qo], positions=range(1, len(qo)+1), patch_artist=True)
axes[1].set_xticks(range(1, len(qo)+1))
axes[1].set_xticklabels([f"Q{q}" for q in qo], fontsize=8)
axes[1].set_ylabel("得分")
axes[1].set_title("各题目二分类得分分布", fontweight="bold")
fig.tight_layout()
fig_paths["fig2"] = save_fig(fig, "fig02.png")

# 图3
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
event_counts = tdf["event_name"].value_counts()
top8 = event_counts.head(8)
pie_data = pd.concat([top8, pd.Series({"其他": event_counts.iloc[8:].sum()})])
ax[0].pie(pie_data.values, labels=pie_data.index, autopct="%1.1f%%", startangle=90, pctdistance=0.82)
ax[0].set_title("事件类型占比", fontweight="bold")
top12 = event_counts.head(12).sort_values()
ax[1].barh(top12.index.astype(str), top12.values, color="#4C78A8")
ax[1].set_xlabel("出现次数")
ax[1].set_title("高频事件类型", fontweight="bold")
fig.tight_layout()
fig_paths["fig3"] = save_fig(fig, "fig03.png")

# 图4
elapsed_sec = adf["total_elapsed_time"] / 1000
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0,0].hist(elapsed_sec, bins=60, color="#4C78A8", edgecolor="white")
axes[0,0].set_title("累计 elapsed_time 分布", fontweight="bold")
axes[0,0].set_xlabel("elapsed_time / 秒")
axes[0,0].set_ylabel("会话数")
axes[0,1].hist(np.log10(elapsed_sec.clip(lower=1)), bins=50, color="#F58518", edgecolor="white")
axes[0,1].set_title("对数变换后分布", fontweight="bold")
axes[0,1].set_xlabel("log10(elapsed_time秒)")
box_data = [elapsed_sec[adf["performance_group"] == g].values for g in adf["performance_group"].cat.categories]
axes[1,0].boxplot(box_data, labels=adf["performance_group"].cat.categories, patch_artist=True)
axes[1,0].tick_params(axis="x", rotation=12, labelsize=8)
axes[1,0].set_ylabel("elapsed_time / 秒")
axes[1,0].set_title("不同表现组累计时间分布", fontweight="bold")
axes[1,1].hist(adf["total_events"], bins=50, color="#54A24B", edgecolor="white")
axes[1,1].axvline(adf["total_events"].median(), color="red", ls="--", label=f"中位数 {adf['total_events'].median():.0f}")
axes[1,1].legend(fontsize=8)
axes[1,1].set_title("每会话事件数分布", fontweight="bold")
axes[1,1].set_xlabel("事件数")
fig.tight_layout()
fig_paths["fig4"] = save_fig(fig, "fig04.png")

# 图5：官方题组难度热力图
q_heat = question_stats.copy()
q_heat["q_group"] = q_heat["question"].map(q_group)
q_heat["display_group"] = pd.Categorical(q_heat["q_group"], categories=["0-4", "5-12", "13-22"], ordered=True)
heat_data = q_heat.pivot_table(index="question", columns="display_group", values="mean_pct", observed=False)
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(heat_data, annot=True, fmt=".1f", cmap="RdYlGn", vmin=25, vmax=100, linewidths=1,
            linecolor="white", cbar_kws={"label": "正确率(%)"}, ax=ax)
ax.set_title("题目官方阶段难度热力图", fontsize=13, fontweight="bold")
ax.set_xlabel("题目所属官方等级组")
ax.set_ylabel("题目编号")
fig.tight_layout()
fig_paths["fig5"] = save_fig(fig, "fig05.png")

# 图6：事件类型占比差异
event_pct_cols = [c for c in adf.columns if c.startswith("event_pct_")]
top_events = [f"event_pct_{e}" for e in event_counts.head(6).index if f"event_pct_{e}" in event_pct_cols]
fig, ax = plt.subplots(figsize=(10, 4.5))
event_by_perf = adf.groupby("performance_group", observed=True)[top_events].mean()
event_by_perf.columns = [c.replace("event_pct_", "") for c in event_by_perf.columns]
event_by_perf.plot(kind="bar", ax=ax, width=0.78)
ax.set_ylabel("平均事件占比(%)")
ax.set_xlabel("")
ax.set_title("不同表现组主要事件类型占比", fontweight="bold")
ax.tick_params(axis="x", rotation=12)
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig_paths["fig6"] = save_fig(fig, "fig06.png")

# 图7：耗时与正确率
adf["time_bucket"] = pd.qcut(elapsed_sec, q=6, labels=[f"Q{i}" for i in range(1,7)], duplicates="drop")
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
sc = axes[0,0].scatter(elapsed_sec, adf["avg_correct"], c=adf["total_events"], s=8, alpha=0.45, cmap="viridis")
axes[0,0].set_xlabel("elapsed_time / 秒")
axes[0,0].set_ylabel("正确率(%)")
axes[0,0].set_title(f"累计时间与正确率 r={time_corr:.3f}", fontweight="bold")
fig.colorbar(sc, ax=axes[0,0], label="事件数")
bucket_data = [adf.loc[adf["time_bucket"] == b, "avg_correct"].values for b in adf["time_bucket"].cat.categories]
axes[0,1].boxplot(bucket_data, labels=adf["time_bucket"].cat.categories, patch_artist=True)
axes[0,1].set_title("耗时分位组正确率", fontweight="bold")
axes[0,1].set_xlabel("Q1最快, Q6最慢")
axes[0,1].set_ylabel("正确率(%)")
axes[1,0].scatter(adf["time_per_event"], adf["avg_correct"], alpha=0.35, s=8, color="#4C78A8")
axes[1,0].set_xlabel("每次操作平均 elapsed_time")
axes[1,0].set_ylabel("正确率(%)")
axes[1,0].set_title("操作节奏与正确率", fontweight="bold")
bucket_mean = adf.groupby("time_bucket", observed=True)["avg_correct"].agg(["mean", "std"])
axes[1,1].errorbar(range(len(bucket_mean)), bucket_mean["mean"], yerr=bucket_mean["std"], fmt="o-", capsize=4, color="#2F4B7C")
axes[1,1].set_xticks(range(len(bucket_mean)))
axes[1,1].set_xticklabels(bucket_mean.index)
axes[1,1].set_title("耗时分位组均值变化", fontweight="bold")
axes[1,1].set_ylabel("平均正确率(%)")
fig.tight_layout()
fig_paths["fig7"] = save_fig(fig, "fig07.png")

# 图8：设置与参与度
setting_stats = {}
for col, label in [("fullscreen_mode", "全屏"), ("hq_mode", "高清"), ("music_mode", "音乐")]:
    g0 = adf.loc[adf[col] == 0, "avg_correct"]
    g1 = adf.loc[adf[col] == 1, "avg_correct"]
    t, p = ttest_ind(g0, g1, equal_var=False)
    setting_stats[label] = {"off_m": g0.mean(), "on_m": g1.mean(), "off_n": len(g0), "on_n": len(g1), "t": t, "p": p}

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (col, label) in zip(axes.flat[:3], [("fullscreen_mode", "全屏"), ("hq_mode", "高清"), ("music_mode", "音乐")]):
    means = adf.groupby(col)["avg_correct"].mean()
    ax.bar(["关闭", "开启"], [means.get(0, np.nan), means.get(1, np.nan)], color=["#B0BEC5", "#26A69A"], edgecolor="white")
    ax.set_ylim(60, 75)
    ax.set_title(f"{label}设置与正确率", fontweight="bold")
    ax.set_ylabel("平均正确率(%)")
axes[1,1].scatter(adf["total_events"], adf["avg_correct"], alpha=0.35, s=8, color="#4C78A8")
axes[1,1].set_title(f"事件数与正确率 r={event_corr:.3f}", fontweight="bold")
axes[1,1].set_xlabel("事件数")
axes[1,1].set_ylabel("正确率(%)")
fig.tight_layout()
fig_paths["fig8"] = save_fig(fig, "fig08.png")

# 图9：等级推进过程
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
level_counts = tdf["level"].value_counts().sort_index()
axes[0,0].bar(level_counts.index, level_counts.values, color="#4C78A8")
axes[0,0].axvspan(-0.5, 4.5, color="#E8F5E9", alpha=0.5)
axes[0,0].axvspan(4.5, 12.5, color="#FFF8E1", alpha=0.5)
axes[0,0].axvspan(12.5, 22.5, color="#FFEBEE", alpha=0.5)
axes[0,0].set_title("各等级事件量", fontweight="bold")
axes[0,0].set_xlabel("等级")
axes[0,0].set_ylabel("事件数")
level_event = tdf.groupby(["level_group", "event_name"], observed=True).size().unstack(fill_value=0)
level_event_pct = level_event.div(level_event.sum(axis=1), axis=0) * 100
level_event_pct[event_counts.head(5).index].plot(kind="bar", stacked=True, ax=axes[0,1], colormap="Set3")
axes[0,1].set_title("等级组事件构成", fontweight="bold")
axes[0,1].set_xlabel("")
axes[0,1].set_ylabel("占比(%)")
axes[0,1].legend(fontsize=6)
stage_means = level_group_stats.loc[["0-4", "5-12", "13-22"], "mean_pct"]
axes[1,0].plot(stage_means.index, stage_means.values, "o-", color="#2E7D32", lw=2)
axes[1,0].set_title("官方题目阶段正确率", fontweight="bold")
axes[1,0].set_ylabel("正确率(%)")
for x, y in zip(stage_means.index, stage_means.values):
    axes[1,0].text(x, y + 1, f"{y:.1f}%", ha="center")
early_qs = [1, 2, 3]
mid_qs = list(range(4, 16))
late_qs = [16, 17, 18]
ecr = labels.loc[labels["question"].isin(early_qs), "correct"].mean() * 100
mcr = labels.loc[labels["question"].isin(mid_qs), "correct"].mean() * 100
lcr = labels.loc[labels["question"].isin(late_qs), "correct"].mean() * 100
bars = axes[1,1].bar(["前期\nQ1-Q3", "中期\nQ4-Q15", "后期\nQ16-Q18"], [ecr, mcr, lcr], color=["#81C784", "#FFD54F", "#FF8A65"])
axes[1,1].set_title("前中后期表现对比", fontweight="bold")
axes[1,1].set_ylabel("正确率(%)")
for b, v in zip(bars, [ecr, mcr, lcr]):
    axes[1,1].text(b.get_x()+b.get_width()/2, v+1, f"{v:.1f}%", ha="center")
fig.tight_layout()
fig_paths["fig9"] = save_fig(fig, "fig09.png")

# 图10：相关性矩阵
corr_cols = [c for c in ["total_events", "total_elapsed_time", "mean_elapsed_time", "std_elapsed_time", "unique_levels",
                         "max_level", "mean_hover", "total_hover", "time_per_event", "avg_correct",
                         "fullscreen_mode", "hq_mode", "music_mode"] if c in adf.columns]
rename = {
    "total_events": "事件总数", "total_elapsed_time": "总elapsed", "mean_elapsed_time": "平均elapsed",
    "std_elapsed_time": "elapsed标准差", "unique_levels": "唯一等级", "max_level": "最高等级",
    "mean_hover": "平均悬停", "total_hover": "总悬停", "time_per_event": "操作间隔",
    "avg_correct": "正确率", "fullscreen_mode": "全屏", "hq_mode": "高清", "music_mode": "音乐",
}
corr = adf[corr_cols].corr().rename(index=rename, columns=rename)
fig, ax = plt.subplots(figsize=(10, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, mask=mask,
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Pearson r", "shrink": 0.75}, ax=ax)
ax.set_title("特征相关性矩阵", fontweight="bold")
fig.tight_layout()
fig_paths["fig10"] = save_fig(fig, "fig10.png")

# 图11：雷达图
radar_cols = ["total_events", "total_elapsed_time", "unique_levels", "max_level", "mean_hover", "time_per_event"]
radar_labels = ["事件总数", "累计时间", "探索等级", "最高等级", "平均悬停", "操作间隔"]
radar = adf.groupby("perform_label")[radar_cols].mean()
radar_norm = (radar - adf[radar_cols].min()) / (adf[radar_cols].max() - adf[radar_cols].min())
angles = np.linspace(0, 2*np.pi, len(radar_cols), endpoint=False).tolist()
angles += angles[:1]
fig = plt.figure(figsize=(7, 6))
ax = plt.subplot(111, polar=True)
for label, color in [("高表现组", "#2CA02C"), ("低表现组", "#D62728")]:
    vals = radar_norm.loc[label].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, color=color, lw=2, label=label)
    ax.fill(angles, vals, color=color, alpha=0.12)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels)
ax.set_yticklabels([])
ax.set_title("高低表现组多维行为画像", fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
fig.tight_layout()
fig_paths["fig11"] = save_fig(fig, "fig11.png")

print("基础图表完成:", len(fig_paths))

基础图表完成: 11


## 5. 高级 EDA：时间、行为路径与空间探索

In [6]:
# 图12：标准化游戏进程分析
advanced = tdf[["session_id", "index", "elapsed_time", "event_name"]].copy()
advanced = advanced.merge(adf[["session_id", "perform_label"]], on="session_id", how="inner")
advanced = advanced.sort_values(["session_id", "index"])
advanced["event_order"] = advanced.groupby("session_id").cumcount()
advanced["session_events"] = advanced.groupby("session_id")["event_order"].transform("max") + 1
advanced["progress_bin"] = pd.cut(
    advanced["event_order"] / advanced["session_events"].clip(lower=1),
    bins=[0, .2, .4, .6, .8, 1.000001],
    labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"],
    include_lowest=True,
)
advanced["delta_time"] = advanced.groupby("session_id")["elapsed_time"].diff().clip(lower=0, upper=300_000)
key_events = event_counts.head(4).index.tolist()
advanced["is_key_event"] = advanced["event_name"].astype(str).isin([str(x) for x in key_events]).astype(int)

time_stage = advanced.groupby(["perform_label", "progress_bin"], observed=True).agg(
    events=("event_name", "size"),
    mean_delta_ms=("delta_time", "mean"),
    key_event_pct=("is_key_event", "mean"),
    sessions=("session_id", "nunique"),
).reset_index()
time_stage["events_per_session"] = time_stage["events"] / time_stage["sessions"]
time_stage["mean_delta_sec"] = time_stage["mean_delta_ms"] / 1000
time_stage["key_event_pct"] *= 100

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for metric, title, ylabel, ax in [
    ("events_per_session", "标准化进程中的事件密度", "每会话事件数", axes[0]),
    ("mean_delta_sec", "平均操作间隔变化", "秒", axes[1]),
    ("key_event_pct", "高频关键事件占比变化", "%", axes[2]),
]:
    sns.lineplot(data=time_stage, x="progress_bin", y=metric, hue="perform_label", marker="o", linewidth=2, ax=ax)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("游戏进程阶段")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=20)
    ax.legend(title="")
fig.tight_layout()
fig_paths["fig12"] = save_fig(fig, "fig12_time_process.png")

print(time_stage.round(2).to_string(index=False))

perform_label progress_bin  events  mean_delta_ms  key_event_pct  sessions  events_per_session  mean_delta_sec
         低表现组        0-20%  154064        1889.90          87.29       601              256.35            1.89
         低表现组       20-40%  153704        1779.19          82.52       601              255.75            1.78
         低表现组       40-60%  153702        2477.60          82.60       601              255.74            2.48
         低表现组       60-80%  153704        1607.42          89.37       601              255.75            1.61
         低表现组      80-100%  153342        1589.47          77.00       601              255.14            1.59
         高表现组        0-20%  146650        1981.59          86.68       736              199.25            1.98
         高表现组       20-40%  146215        1657.89          84.37       736              198.66            1.66
         高表现组       40-60%  146211        2661.84          82.37       736              198.66            2.66
 

In [7]:
# 图13：事件转移矩阵
top_transition_events = event_counts.head(8).index.astype(str).tolist()
seq = tdf[["session_id", "index", "event_name"]].copy()
seq = seq.merge(adf[["session_id", "perform_label"]], on="session_id", how="inner")
seq = seq.sort_values(["session_id", "index"])
seq["event_name"] = seq["event_name"].astype(str)
seq["event_s"] = np.where(seq["event_name"].isin(top_transition_events), seq["event_name"], "other")
seq["next_event"] = seq.groupby("session_id")["event_s"].shift(-1)
seq = seq.dropna(subset=["next_event"])

def transition_matrix(label):
    sub = seq.loc[seq["perform_label"] == label]
    mat = pd.crosstab(sub["event_s"], sub["next_event"]).reindex(index=top_transition_events + ["other"], columns=top_transition_events + ["other"], fill_value=0)
    mat = mat.div(mat.sum(axis=1).replace(0, np.nan), axis=0) * 100
    return mat.fillna(0)

high_mat = transition_matrix("高表现组")
low_mat = transition_matrix("低表现组")
diff_mat = high_mat - low_mat

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, mat, title, cmap, center in [
    (axes[0], high_mat, "高表现组事件转移概率(%)", "YlGnBu", None),
    (axes[1], low_mat, "低表现组事件转移概率(%)", "YlOrRd", None),
    (axes[2], diff_mat, "高-低表现组转移差异(百分点)", "RdBu_r", 0),
]:
    sns.heatmap(mat, ax=ax, cmap=cmap, center=center, linewidths=.4, linecolor="white",
                cbar_kws={"shrink": .75}, xticklabels=True, yticklabels=True)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("下一事件")
    ax.set_ylabel("当前事件")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)
fig.tight_layout()
fig_paths["fig13"] = save_fig(fig, "fig13_transition_matrix.png")

top_diff = diff_mat.stack().sort_values(key=lambda s: s.abs(), ascending=False).head(8)
print("高低表现组事件转移差异最大的路径（百分点）")
print(top_diff.round(2).to_string())

高低表现组事件转移差异最大的路径（百分点）
event_s         next_event        
object_click    object_click         -12.34
                navigate_click         5.47
                notification_click     5.02
map_hover       other                  4.90
                map_hover             -4.90
navigate_click  navigate_click        -3.42
other           navigate_click         3.26
person_click    person_click           2.35


In [8]:
# 图14：空间交互密度
coord = tdf[["session_id", "screen_coor_x", "screen_coor_y", "room_coor_x", "room_coor_y", "event_name"]].copy()
coord = coord.merge(adf[["session_id", "perform_label"]], on="session_id", how="inner")
coord = coord.dropna(subset=["screen_coor_x", "screen_coor_y"])

# 去除明显极端坐标，避免少数异常值拉伸画布
x1, x99 = coord["screen_coor_x"].quantile([0.01, 0.99])
y1, y99 = coord["screen_coor_y"].quantile([0.01, 0.99])
coord_clip = coord[(coord["screen_coor_x"].between(x1, x99)) & (coord["screen_coor_y"].between(y1, y99))].copy()

space_stats = coord_clip.groupby(["perform_label", "session_id"]).agg(
    x_std=("screen_coor_x", "std"),
    y_std=("screen_coor_y", "std"),
    x_range=("screen_coor_x", lambda s: s.max() - s.min()),
    y_range=("screen_coor_y", lambda s: s.max() - s.min()),
    n_points=("screen_coor_x", "size"),
).reset_index()
space_summary = space_stats.groupby("perform_label")[["x_std", "y_std", "x_range", "y_range", "n_points"]].mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, label, cmap in [(axes[0], "高表现组", "Greens"), (axes[1], "低表现组", "Reds")]:
    sub = coord_clip.loc[coord_clip["perform_label"] == label]
    hb = ax.hexbin(sub["screen_coor_x"], sub["screen_coor_y"], gridsize=45, cmap=cmap, mincnt=1, bins="log")
    ax.invert_yaxis()
    ax.set_title(f"{label}屏幕交互密度", fontweight="bold")
    ax.set_xlabel("screen_coor_x")
    ax.set_ylabel("screen_coor_y")
    fig.colorbar(hb, ax=ax, label="log(事件数)")

space_summary[["x_range", "y_range"]].plot(kind="bar", ax=axes[2], color=["#4C78A8", "#F58518"], edgecolor="white")
axes[2].set_title("空间探索范围均值", fontweight="bold")
axes[2].set_ylabel("坐标范围")
axes[2].tick_params(axis="x", rotation=0)
axes[2].legend(["横向范围", "纵向范围"], fontsize=8)
fig.tight_layout()
fig_paths["fig14"] = save_fig(fig, "fig14_spatial_density.png")

print("空间探索指标")
print(space_summary.round(2).to_string())

空间探索指标
                    x_std       y_std  x_range  y_range  n_points
perform_label                                                    
低表现组           234.660004  120.110001   843.30   575.74   1117.92
高表现组           231.089996  117.529999   842.37   574.69    878.48


## 6. 自动生成修订版 Word 报告

In [9]:
def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)

def set_cell_text(cell, text, bold=False, size=9.5):
    cell.text = ""
    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run(str(text))
    run.font.name = "宋体"
    run.element.rPr.rFonts.set(qn("w:eastAsia"), "宋体")
    run.font.size = Pt(size)
    run.bold = bold
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

def add_heading(doc, text, level=1):
    h = doc.add_heading(text, level=level)
    for r in h.runs:
        r.font.name = "黑体"
        r.element.rPr.rFonts.set(qn("w:eastAsia"), "黑体")
        r.font.size = Pt({0: 20, 1: 14, 2: 12, 3: 11}.get(level, 11))
        r.font.color.rgb = RGBColor(0, 51, 102) if level <= 1 else RGBColor(40, 40, 40)
    return h

def add_para(doc, text, bold=False, size=10.5, align=None, indent=True, after=3):
    p = doc.add_paragraph()
    if indent and align is None:
        p.paragraph_format.first_line_indent = Pt(20)
    if align is not None:
        p.alignment = align
    p.paragraph_format.line_spacing = 1.25
    p.paragraph_format.space_after = Pt(after)
    run = p.add_run(text)
    run.font.name = "宋体"
    run.element.rPr.rFonts.set(qn("w:eastAsia"), "宋体")
    run.font.size = Pt(size)
    run.bold = bold
    return p

def add_figure(doc, path, caption, width=4.4):
    if Path(path).exists():
        p = doc.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.paragraph_format.space_before = Pt(6)
        p.paragraph_format.space_after = Pt(1)
        p.add_run().add_picture(path, width=Inches(width))
        cap = doc.add_paragraph()
        cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
        cap.paragraph_format.space_after = Pt(5)
        r = cap.add_run(caption)
        r.font.name = "宋体"
        r.element.rPr.rFonts.set(qn("w:eastAsia"), "宋体")
        r.font.size = Pt(9)
        r.italic = True

def add_table(doc, headers, rows):
    table = doc.add_table(rows=1, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.style = "Table Grid"
    for i, h in enumerate(headers):
        set_cell_text(table.rows[0].cells[i], h, bold=True)
        set_cell_shading(table.rows[0].cells[i], "D9EAF7")
    for row in rows:
        cells = table.add_row().cells
        for i, val in enumerate(row):
            set_cell_text(cells[i], val)
    doc.add_paragraph()
    return table

def fmt_p(p):
    if p < 0.001:
        return "<0.001"
    return f"={p:.4f}"

doc = Document()
sec = doc.sections[0]
sec.top_margin = Inches(0.85)
sec.bottom_margin = Inches(0.85)
sec.left_margin = Inches(0.9)
sec.right_margin = Inches(0.9)

# cover
for _ in range(4):
    doc.add_paragraph()
p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run("基于游戏行为数据的\n学生表现高级EDA分析研究")
r.font.name = "黑体"; r.element.rPr.rFonts.set(qn("w:eastAsia"), "黑体")
r.font.size = Pt(25); r.bold = True; r.font.color.rgb = RGBColor(0, 51, 102)
doc.add_paragraph()
p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run("——数据分析与可视化课程期末报告")
r.font.name = "黑体"; r.element.rPr.rFonts.set(qn("w:eastAsia"), "黑体")
r.font.size = Pt(16); r.font.color.rgb = RGBColor(80, 80, 80)
for _ in range(3):
    doc.add_paragraph()
for line in ["学    院：信息与商务管理学院", "专    业：大数据管理与应用", "学    号：2023110721", "姓    名：海金锁", "提交日期：2026年6月"]:
    add_para(doc, line, size=13, align=WD_ALIGN_PARAGRAPH.CENTER, indent=False, after=1)
doc.add_page_break()

hardest = question_sorted.iloc[0]
easiest = question_sorted.iloc[-1]
lg0 = level_group_stats.loc["0-4", "mean_pct"]
lg1 = level_group_stats.loc["5-12", "mean_pct"]
lg2 = level_group_stats.loc["13-22", "mean_pct"]

add_heading(doc, "摘  要", 1)
add_para(doc, f"本研究基于 Kaggle 竞赛 Predict Student Performance from Game Play 的教育游戏日志数据，围绕学生答题表现差异展开数据整理、清洗、描述性统计、探索性分析和可视化展示。数据包含约2600万行训练日志和{len(labels):,}条答题标签；考虑本地计算资源，本研究读取前{SAMPLE_SIZE//10000}万行日志并剔除边界截断会话，最终得到{len(adf):,}个可用于行为分析的有效会话。")
add_para(doc, "本研究的重点不是构建复杂预测模型，而是提出清楚的分析问题，并用合适的图表展示数据分布、变量关系、类别差异、时间变化和空间特征。除常规题目难度、事件类型、耗时、相关性分析外，本版进一步加入标准化游戏进程、事件转移矩阵和空间交互密度三个高级 EDA 视角，将分析从静态结果描述扩展到过程性行为解释。")
add_para(doc, f"主要发现包括：最难题目为Q{int(hardest['question'])}，正确率{hardest['mean_pct']:.1f}%；最易题目为Q{int(easiest['question'])}，正确率{easiest['mean_pct']:.1f}%。官方题目阶段正确率分别为0-4级{lg0:.1f}%、5-12级{lg1:.1f}%、13-22级{lg2:.1f}%，组间差异显著（ANOVA F={level_f:.2f}, p{fmt_p(level_p)}）。事件总数与正确率呈显著负相关（r={event_corr:.3f}, p{fmt_p(event_p)}），说明简单以“操作更多”衡量学习投入并不充分；高级 EDA 进一步显示，高低表现学生在进程节奏、事件路径和空间探索范围上存在可观察差异。")
add_para(doc, "关键词：数据分析；高级EDA；教育游戏；学生表现；时间维度；空间可视化", bold=True)
doc.add_page_break()

add_heading(doc, "目  录", 1)
for item in [
    "一、研究背景与问题",
    "二、数据来源与预处理",
    "三、描述性统计分析",
    "四、探索性可视化分析",
    "五、变量关系与高级EDA分析",
    "六、结论与讨论",
    "参考文献",
]:
    add_para(doc, item, indent=False)
doc.add_page_break()

add_heading(doc, "一、研究背景与问题", 1)
add_heading(doc, "1.1 研究背景", 2)
add_para(doc, "教育游戏能够在学习过程中自动记录学生的每一次操作，包括点击、导航、悬停、对话和场景切换等事件。这类日志不同于传统考试成绩，它不仅包含最终答题结果，还包含学生在学习过程中的行为轨迹，因此适合通过探索性数据分析和可视化方法进行过程性学习分析。")
add_para(doc, "本研究使用的数据来自教育游戏《Jo Wilder and the Capitol Case》。学生在游戏中通过探索场景、与角色互动、收集线索并回答问题推进剧情。数据具有典型的多维日志特征：包含时间戳、事件类型、等级组、屏幕坐标和答题标签，适合从结果、行为、时间和空间多个角度进行分析。")
add_heading(doc, "1.2 研究问题", 2)
for i, q in enumerate([
    "18道题目的正确率分布有何特征，哪些题目最难、哪些题目最容易？",
    "高表现学生与低表现学生在事件类型构成和参与度指标上是否存在差异？",
    "游戏累计时间、操作节奏与答题正确率之间存在怎样的关系？",
    "将游戏过程标准化为若干阶段后，高低表现学生的行为节奏是否不同？",
    "学生的事件转移路径和空间交互密度是否能揭示更细的过程性差异？",
], 1):
    add_para(doc, f"问题{i}：{q}")
add_heading(doc, "1.3 研究目的", 2)
add_para(doc, "本研究旨在通过描述性统计、探索性分析和可视化展示，对学生游戏行为数据进行解释性分析。研究不以 Kaggle 排名或复杂模型为目标，而是关注能否将原始日志整理为清楚的分析对象，并用恰当图表回答一开始提出的问题。")
add_para(doc, "具体而言，报告采用“结果表现—行为特征—过程维度”的分析框架：首先用答题正确率刻画学生表现和题目难度，其次用事件类型、事件数量和时间指标解释行为差异，最后进一步加入标准化进程、事件转移和空间坐标三个过程性视角。这样的结构既能覆盖课程要求中的数据分布、变量关系和类别差异，也能体现时间变化与空间特征。")

add_heading(doc, "二、数据来源与预处理", 1)
add_heading(doc, "2.1 数据来源", 2)
add_para(doc, f"数据集包括 train.csv、train_labels.csv、test.csv 和 sample_submission.csv。其中 train.csv 是游戏事件日志，包含约2600万行记录；train_labels.csv 包含{len(labels):,}条答题标签，覆盖{labels['session'].nunique():,}个会话，每个会话对应18道题。")
add_heading(doc, "2.2 数据整理与清洗", 2)
add_para(doc, f"由于原始日志文件约4.7GB，本研究采用前{SAMPLE_SIZE//10000}万行日志作为行为分析样本。为避免采样边界造成一个会话被截断，剔除了最后一个不完整会话。读取时将事件名、文本、等级组等字段转为 category，将坐标字段转为 float32，将设置字段转为 int8，从而降低内存占用。")
add_para(doc, "标签字段 session_id 形如“数字ID_q题号”，本研究使用正则表达式拆分出纯会话ID和题号，并根据竞赛题目设置将Q1-Q3划为0-4级、Q4-Q13划为5-12级、Q14-Q18划为13-22级。需要注意：题目难度分析基于全量标签，行为模式和时空分析基于抽样日志内能够匹配到标签的会话。")
add_heading(doc, "2.3 数据预处理详细流程", 2)
for step in [
    "第一步，读取原始数据文件。使用pandas读取train.csv中的前150万行游戏日志，并读取完整的train_labels.csv标签文件。由于日志文件规模较大，在读取阶段即指定字段类型，减少内存占用并提高后续分组运算效率。",
    "第二步，处理采样边界问题。按行截取前150万行时，最后一个session_id对应的会话可能只保留了部分事件。为避免不完整会话低估事件数、耗时和空间轨迹，本研究识别并删除采样边界处的最后一个截断会话。",
    "第三步，解析标签字段。train_labels.csv中的session_id同时包含会话编号和题号，例如“20090109393214576_q1”。本研究通过正则表达式拆分出纯会话ID和question题号，并将题号转换为整数，便于按题目、阶段和会话进行统计。",
    "第四步，构造题目阶段变量。根据竞赛题目设置，将Q1-Q3划分为0-4级阶段，将Q4-Q13划分为5-12级阶段，将Q14-Q18划分为13-22级阶段。该变量主要用于题目难度、阶段差异和前中后期表现分析。",
    "第五步，处理缺失值与异常值。hover_duration只在悬停类事件中出现，缺失具有结构性，因此在会话聚合时保留其含义并在最终特征表中填充为0；坐标分析中则删除缺失坐标，并使用1%和99%分位数过滤极端坐标，避免少数异常点拉伸空间图。",
    "第六步，进行会话级特征聚合。以session_id为单位统计事件总数、累计elapsed_time、平均elapsed_time、elapsed_time标准差、探索等级数、最高等级、平均悬停时长、总悬停时长，并计算各事件类型在会话中的占比。",
    "第七步，合并行为特征与答题表现。先根据全量标签计算每个会话18道题的平均正确率、答题数量和答对数量，再与抽样日志得到的会话级行为特征按session_id连接，形成最终用于行为分析的数据表。",
    "第八步，构造分组变量。根据会话平均正确率划分四个表现组，并以中位数为界构造高表现组和低表现组。前者用于描述不同水平区间的总体差异，后者用于事件转移矩阵、空间密度和高级EDA中的对比分析。",
]:
    add_para(doc, step)

add_heading(doc, "2.4 会话级特征工程", 2)
add_para(doc, "从原始日志中聚合得到会话级特征，包括事件总数、累计 elapsed_time、平均 elapsed_time、elapsed_time 标准差、探索等级数、最高等级、平均悬停时长、总悬停时长、各事件类型占比、全屏/高清/音乐设置众数，以及每次操作平均 elapsed_time。随后与会话平均正确率合并，形成最终行为分析数据集。")
add_heading(doc, "2.5 分析方法与可视化设计", 2)
add_para(doc, "本研究使用的分析方法分为四类。第一类是描述性统计，用于概括正确率、事件数、耗时和题目难度等基础特征；第二类是分组比较，用于比较高表现组和低表现组在行为指标上的差异；第三类是相关性与显著性检验，包括Pearson相关、t检验、卡方检验和ANOVA，用于判断观察到的差异是否具有统计支持；第四类是高级探索性分析，包括标准化进程分析、事件转移矩阵和空间交互密度分析，用于展示学生在游戏过程中的时间节奏、操作路径和空间探索模式。")
add_para(doc, "可视化设计遵循“图表服务问题”的原则：饼图和直方图用于展示总体分布，排序柱状图和热力图用于呈现题目难度差异，箱线图和散点图用于比较分布和变量关系，相关矩阵用于总览多变量关系，雷达图用于压缩展示行为画像，进程折线图、转移矩阵和空间密度图则用于体现时间、路径和空间三个更高级的过程维度。")

add_heading(doc, "三、描述性统计分析", 1)
add_heading(doc, "3.1 核心统计指标", 2)
rows = [
    ["答题标签记录数", f"{len(labels):,}", "全量 train_labels.csv"],
    ["标签会话数", f"{labels['session'].nunique():,}", "每会话18道题"],
    ["抽样日志行数", f"{len(tdf):,}", "剔除边界截断会话后"],
    ["行为分析会话数", f"{len(adf):,}", "日志与标签匹配"],
    ["整体答题正确率", f"{overall_correct:.1f}%", "全量标签"],
    ["会话平均正确率", f"{mean_correct:.1f}%", "抽样行为会话"],
    ["会话正确率中位数", f"{median_correct:.1f}%", "用于划分高低表现组"],
    ["事件数与正确率相关", f"r={event_corr:.3f}, p{fmt_p(event_p)}", "Pearson相关"],
    ["累计elapsed与正确率相关", f"r={time_corr:.3f}, p{fmt_p(time_p)}", "Pearson相关"],
]
add_table(doc, ["指标", "数值", "说明"], rows)
add_figure(doc, fig_paths["fig1"], "图1  答题正确率总体分布与会话正确率直方图", 4.3)
add_para(doc, "图1显示，整体答题正确率约为70.6%，会话级正确率集中在中高区间，但仍存在明显个体差异。这说明仅看平均正确率不足以解释学生表现差异，需要进一步从题目难度、行为类型和过程特征展开分析。")
add_para(doc, "从课程分析角度看，图1的作用是先建立“结果变量”的基本背景：如果所有学生正确率都非常接近，后续行为差异分析的意义会较弱；而当前分布既有集中趋势，也有较明显离散性，说明数据中确实存在值得解释的表现差异。后文的行为、时间和空间分析，都是围绕这种差异展开。")
add_para(doc, "进一步观察会话正确率直方图可以发现，学生表现并不是均匀分散在0%到100%之间，而是主要集中在中高分区间，同时左侧仍保留一部分低正确率会话。这种形态说明游戏任务对多数学生具有可完成性，但仍能区分出明显的困难群体。因此，后续将会话按中位数划分为高表现组和低表现组具有实际意义。")

add_heading(doc, "3.2 题目难度分析", 2)
add_figure(doc, fig_paths["fig2"], "图2  各题目正确率排序与得分分布箱线图", 4.6)
add_para(doc, f"从题目层面看，最难题目为Q{int(hardest['question'])}，正确率仅{hardest['mean_pct']:.1f}%；最易题目为Q{int(easiest['question'])}，正确率为{easiest['mean_pct']:.1f}%，二者相差{easiest['mean_pct']-hardest['mean_pct']:.1f}个百分点。中等难度题目更能体现学生差异，而极易题和极难题更多反映基础掌握或综合挑战。")
add_para(doc, "这里同时使用排序柱状图和箱线图，是为了避免只用单一均值判断题目质量。排序柱状图便于快速识别难题和易题，箱线图则提醒我们每道题本质上是0/1二分类结果，分布形态与连续成绩不同。因此，报告中对题目难度的解释主要基于正确率差异，而不把箱线图中的离散形态误读为连续变量波动。")
add_para(doc, "从教育测量角度看，极易题能够检验学生是否掌握最基础的信息，极难题则可能反映综合推理、线索整合或题目表述难度。真正对区分学生水平更有帮助的，往往是正确率处于中间区间的题目，因为这些题目既不会让绝大多数学生全部答对，也不会让大多数学生全部失败。")

add_heading(doc, "3.3 事件类型与时间分布", 2)
add_figure(doc, fig_paths["fig3"], "图3  游戏事件类型分布", 4.4)
add_para(doc, "事件类型分布高度集中，少数高频事件构成了大部分游戏行为。这说明后续行为分析不应只看总事件数，而应关注事件类型构成和事件之间的转移关系。")
add_para(doc, "事件分布的集中性也反映了教育游戏数据的一个特点：日志数量很大，但真正能解释学习过程的行为类型并不一定同样多。若直接把所有事件视为等价操作，容易把导航、对话、观察等不同含义混在一起。因此，本研究后续用事件占比和事件转移矩阵补充总量指标，使分析从“操作了多少次”进一步转向“主要做了什么、下一步通常做什么”。")
add_para(doc, "从图形表达上看，饼图适合展示总体构成，横向条形图更适合比较不同事件类型之间的数量差距。两者结合后可以看出，游戏行为日志并非由大量均衡事件构成，而是由若干核心交互行为主导。这为后续筛选主要事件、构建事件转移矩阵提供了依据。")
add_figure(doc, fig_paths["fig4"], "图4  累计elapsed_time与事件数量分布", 4.6)
add_para(doc, "elapsed_time 字段是游戏日志中的累计时间指标，存在较长间隔和极端值，因此本研究不将其简单解释为真实连续游戏时长，而是结合对数变换、分位组和操作间隔进行相对比较。")
add_para(doc, "图4说明时间类变量需要谨慎处理。原始 elapsed_time 呈明显右偏，少数会话的数值会拉高均值，如果直接用平均值描述“学生玩了多久”，结论容易失真。因此报告更重视中位数、分位组和对数变换后的形态，并在解释中把它作为相对节奏指标，而不是绝对学习时长。")
add_para(doc, "图4中的事件数量分布同样呈现右偏特征，说明少数会话包含非常多的操作记录。这类会话可能对应深度探索，也可能对应重复尝试或无效点击，因此事件数本身具有双重含义。正因为如此，后续分析没有直接把事件数定义为“学习投入”，而是继续结合正确率、事件类型和行为路径进行解释。")

add_heading(doc, "四、探索性可视化分析", 1)
add_heading(doc, "4.1 题目阶段差异", 2)
add_figure(doc, fig_paths["fig5"], "图5  题目官方阶段难度热力图", 4.2)
add_para(doc, f"图5按题目所属官方阶段展示难度。0-4级题目平均正确率为{lg0:.1f}%，5-12级为{lg1:.1f}%，13-22级为{lg2:.1f}%。ANOVA结果为F={level_f:.2f}, p{fmt_p(level_p)}，说明不同阶段题目正确率存在显著差异。值得注意的是，后期题目并非单调变难，Q18正确率较高，说明题目难度不仅由阶段决定，也与具体题目内容有关。")
add_para(doc, "这一结果对游戏教学设计有一定启发：阶段提升通常意味着任务复杂度增加，但具体题目仍可能受到提示充分程度、题干清晰度和知识点熟悉度影响。中期阶段正确率下降最明显，说明学生在从入门操作转向更复杂探索时可能遇到门槛；如果用于教学干预，这一阶段比单纯的后期阶段更值得重点关注。")
add_para(doc, "热力图的优势在于能够用颜色深浅快速定位异常题目和阶段差异。图中中期阶段出现多道偏难题，说明该阶段可能承担了更多知识整合或推理任务；而后期并非全部呈低正确率，则说明游戏后期可能存在复习、总结或提示更明确的题目。这种结果比单纯按题号顺序判断难度更可靠。")
add_heading(doc, "4.2 行为类型差异", 2)
add_figure(doc, fig_paths["fig6"], "图6  不同表现组主要事件类型占比对比", 4.5)
add_para(doc, "图6显示，不同表现组在高频事件类型占比上存在可观察差异。与单纯统计事件总数相比，事件结构更能体现学生在游戏过程中的操作偏好和学习策略差异。")
add_para(doc, "需要注意的是，事件类型占比本身不能直接说明某类操作“导致”高分或低分。更合理的解释是：不同表现水平的学生在完成同一游戏任务时，可能采用了不同的探索方式。例如，高表现学生的操作可能更集中在有效信息获取和任务推进上，低表现学生则可能包含更多重复导航或试错操作。这种解释与后文事件转移矩阵形成互补。")
add_para(doc, "分组柱状图的价值在于把“表现水平”这一结果变量和“事件结构”这一过程变量放在同一视图中比较。若某类事件在低表现组占比更高，不能立即说明该事件有负面作用，但可以提示该类行为可能与低效探索或任务困难有关；若某类事件在高表现组中更稳定，则可能代表更有效的信息获取路径。")
add_heading(doc, "4.3 时间与表现关系", 2)
add_figure(doc, fig_paths["fig7"], "图7  累计elapsed_time、操作节奏与正确率关系", 4.7)
add_para(doc, f"累计 elapsed_time 与正确率的线性相关较弱（r={time_corr:.3f}, p{fmt_p(time_p)}），说明“花费时间越长表现越好”并不成立。分位组图更适合揭示非线性关系：过快或过慢的会话都可能对应较低表现，适度、稳定的操作节奏更值得关注。")
add_para(doc, "这一部分体现了探索性分析的价值：如果只计算一个线性相关系数，容易得出“时间关系不明显”的简单结论；而分位组和箱线图揭示出更细的模式，即时间投入可能存在适度区间。过快可能意味着浏览不充分或跳过线索，过慢则可能对应卡顿、犹豫或反复尝试。该解释虽然不能作为因果结论，但能为教师观察学生学习状态提供更具体的线索。")
add_para(doc, "图7使用散点图、箱线图和均值误差线同时呈现时间与表现关系，是因为单一图表很难完整描述非线性模式。散点图展示整体分散程度，箱线图比较不同耗时组的分布差异，误差线则展示组间均值走势。三种视角结合后，可以更稳妥地说明时间变量与表现之间不是简单正相关关系。")
add_heading(doc, "4.4 设置变量与参与度", 2)
add_figure(doc, fig_paths["fig8"], "图8  游戏设置、事件数与答题表现关系", 4.7)
add_para(doc, f"全屏、高清和音乐设置与正确率的差异均不强，更多可能反映设备和个人偏好。事件总数与正确率呈显著负相关（r={event_corr:.3f}, p{fmt_p(event_p)}），说明更多操作并不一定意味着更好学习效果，低表现学生可能存在反复尝试或低效操作。")
add_para(doc, "这个发现也是本报告中比较重要的反直觉结果。通常我们会把“参与度高”理解为操作更多、停留更久，但在游戏学习场景中，过多操作可能并不代表有效投入，反而可能表示学生在任务中迷失、反复寻找或重复点击。因此，本研究后续把参与度进一步拆解为行为结构、阶段节奏和空间范围，而不是简单使用事件总数作为唯一指标。")
add_para(doc, "图8还说明，游戏环境设置变量的解释力有限。全屏、高清和音乐是否开启，可能受到设备、浏览器状态或个人习惯影响，并不一定反映学生是否更认真学习。相比之下，事件数与正确率之间的关系更明显，但其含义仍需结合行为类型解释，这也是本研究继续分析行为路径和空间密度的原因。")
add_heading(doc, "4.5 等级推进变化", 2)
add_figure(doc, fig_paths["fig9"], "图9  等级推进过程中的事件构成与阶段表现", 4.7)
add_para(doc, f"从前中后期看，Q1-Q3平均正确率为{ecr:.1f}%，Q4-Q15为{mcr:.1f}%，Q16-Q18为{lcr:.1f}%。中期题目明显更具挑战性，后期正确率回升，说明游戏难度并非简单随等级单调增加，而是具有阶段性波动。")
add_para(doc, "等级推进图把题目表现和游戏结构联系起来。前期题目正确率高，可能与新手引导、任务目标明确有关；中期下降说明学生开始面对更复杂的信息整合；后期回升则可能是因为学生逐渐熟悉游戏机制，或者后期题目本身提示更明确。由此可见，阶段分析比单个题目分析更能体现学习过程中的变化。")
add_para(doc, "从事件构成角度看，不同等级组中的主要操作类型并不完全一致，这说明游戏过程本身具有阶段性：前期更偏向熟悉界面和基础交互，中期可能出现更多探索和信息收集，后期则可能进入总结或任务收束。将事件构成和题目正确率放在一起分析，可以更好地解释为什么某些阶段表现会下降。")

add_heading(doc, "五、变量关系与高级EDA分析", 1)
add_heading(doc, "5.1 特征相关性矩阵", 2)
add_figure(doc, fig_paths["fig10"], "图10  特征相关性矩阵", 4.7)
add_para(doc, "相关性矩阵显示，时间类指标之间存在较强相关，事件总数与正确率之间的关系最突出。但相关性只能反映线性关系，不能直接说明因果，因此需要结合过程维度进一步分析。")
add_para(doc, "相关性矩阵的另一个作用是帮助识别变量冗余。例如累计时间、平均时间和操作间隔之间往往存在较强关联，若后续进行建模或聚类，需要避免把高度相似的变量重复纳入解释。对于本课程报告而言，相关性矩阵主要承担“总览变量关系”的功能，为后续选择重点解释变量提供依据。")
add_para(doc, "在读图时需要重点关注两类关系：一类是行为变量与正确率之间的关系，另一类是行为变量彼此之间的关系。前者帮助回答哪些行为特征与表现有关，后者帮助判断变量是否重复表达了相似信息。通过这种矩阵式呈现，读者可以快速把握多变量之间的整体结构，而不必逐一查看所有散点图。")
add_heading(doc, "5.2 高低表现组行为画像", 2)
add_figure(doc, fig_paths["fig11"], "图11  高低表现组多维行为画像雷达图", 4.0)
add_para(doc, "雷达图将多项行为指标压缩到一个视图中，显示高低表现组并不是在单一维度上差异明显，而是在事件数、操作间隔、悬停等多个维度上共同形成不同画像。")
add_para(doc, "雷达图的优势在于整体感强，适合用于报告中的综合比较；但它也有局限：不同维度经过归一化后只能比较相对高低，不能直接比较原始单位。因此本研究没有只依赖雷达图得出结论，而是把它作为对前面统计表、散点图和相关矩阵的概括性补充。")
add_para(doc, "从图11可以把高低表现组理解为两种不同的行为画像，而不是简单的“好学生”和“差学生”。高表现组可能表现出更高的操作效率或更集中的行为策略，低表现组则可能在事件数量、悬停或操作间隔上呈现不同模式。雷达图的意义在于提供整体轮廓，为后续时间和空间分析提供过渡。")
add_heading(doc, "5.3 统计检验补充", 2)
for label, res in setting_stats.items():
    sig = "显著" if res["p"] < 0.05 else "未达显著"
    add_para(doc, f"{label}设置：关闭组n={res['off_n']:,}，平均正确率{res['off_m']:.1f}%；开启组n={res['on_n']:,}，平均正确率{res['on_m']:.1f}%；t={res['t']:.2f}, p{fmt_p(res['p'])}，{sig}。")
add_para(doc, "这些检验结果支持一个重要判断：环境设置变量对表现的解释力有限，行为过程变量更值得进一步挖掘。")

add_heading(doc, "5.4 时间-行为-空间综合分析", 2)
add_figure(doc, fig_paths["fig12"], "图12  标准化游戏进程中的行为节奏变化", 4.9)
add_para(doc, "图12将每个会话按事件顺序标准化为五个阶段，使不同总时长的学生具有可比性。结果显示，高低表现组在事件密度、平均操作间隔和关键事件占比上呈现不同变化轨迹。该图的价值在于从“总共做了多少”转向“在游戏不同阶段如何行动”，更符合过程性学习分析思路。")
add_para(doc, "标准化进程分析是本报告的高级 EDA 重点之一。它不要求所有学生拥有相同游戏时长，而是把每个会话内部的事件序列压缩到统一进度轴上，从而比较“早期—中期—后期”的行为节奏。这样的处理能减少个体总时长差异带来的干扰，也更接近教师实际关心的问题：学生是在一开始就迷失，还是在中后期出现效率下降。")
add_para(doc, "从图12的三个子图可以分别观察操作密度、操作间隔和关键事件占比。若某一组在全程都保持更高事件密度，说明其操作更频繁；若中期操作间隔明显拉长，可能意味着学生在该阶段需要更多思考或遇到困难；若关键事件占比在后期下降，则可能说明学生开始偏离主要任务路径。")
add_figure(doc, fig_paths["fig13"], "图13  高低表现组事件转移矩阵对比", 5.4)
add_para(doc, "图13进一步从行为顺序角度分析事件之间的转移概率。与只比较事件占比不同，转移矩阵关注“当前操作之后通常发生什么”。高低表现组在若干事件路径上的差异说明，学生表现差异可能体现在操作链条和探索路径上，而不只是事件数量。")
add_para(doc, "事件转移矩阵可以理解为一种简化的行为路径分析。普通柱状图只能告诉我们某类事件出现得多不多，而转移矩阵能够展示事件之间的连接关系。例如，如果某组学生更频繁地在导航和重复点击之间循环，可能说明他们在场景中寻找目标的效率较低；如果另一组学生更稳定地从观察、对话转向任务推进，则说明其行为链条更清晰。")
add_para(doc, "图13右侧的差异矩阵尤其值得关注，因为它直接展示高表现组相对于低表现组在哪些转移路径上更高或更低。颜色接近中性表示两组路径相似，颜色差异明显则提示该路径可能具有区分作用。虽然该图不能直接说明哪条路径导致高分，但可以帮助识别后续深入分析的重点行为链。")
add_figure(doc, fig_paths["fig14"], "图14  高低表现组空间交互密度对比", 5.0)
add_para(doc, "图14基于 screen_coor_x 和 screen_coor_y 展示空间交互密度，并比较空间探索范围。空间图使日志中的坐标字段得到充分利用：如果某组学生交互更集中，可能说明其操作路径更固定；如果覆盖范围更广，可能反映更充分的探索。该结论仍属于探索性发现，需要结合游戏场景内容进一步解释。")
add_para(doc, "空间维度的加入使报告不再局限于表格和时间序列。教育游戏中的坐标点能够反映学生注意力和操作位置的分布，高低表现组的密度差异可以帮助识别学生是否集中在关键区域、是否存在大范围无效探索。不过，由于本研究没有直接还原每个房间的地图结构，因此空间图主要用于发现模式，而不是精确判断某个坐标点的教学含义。")
add_para(doc, "从展示形式看，hexbin密度图比普通散点图更适合处理大量坐标点，因为它可以减少点重叠带来的视觉干扰。右侧的空间范围比较则补充了密度图的整体判断：如果某组横向或纵向范围更大，说明其操作覆盖区域更广；如果范围较小且密度集中，则说明交互位置更固定。这为理解学生探索方式提供了空间证据。")

add_heading(doc, "六、结论与讨论", 1)
add_heading(doc, "6.1 主要结论", 2)
add_para(doc, f"第一，题目难度存在明显差异。全量标签数据显示，Q{int(hardest['question'])}正确率最低，仅为{hardest['mean_pct']:.1f}%；Q{int(easiest['question'])}正确率最高，达到{easiest['mean_pct']:.1f}%。按官方题目阶段划分，0-4级、5-12级和13-22级的正确率分别为{lg0:.1f}%、{lg1:.1f}%和{lg2:.1f}%，且阶段间差异显著。这说明游戏题目并非均匀分布在同一难度层级上，中期题目对学生形成了更明显的挑战。")
add_para(doc, "第二，学生表现差异不能只用“做了多少操作”解释。事件总数与正确率呈显著负相关，这一结果提示我们，更多操作并不必然代表更高参与度或更好学习效果。低表现学生可能存在重复尝试、无效导航或反复点击等行为，而高表现学生可能以更少但更有效的操作完成任务。因此，对教育游戏日志的解释应关注行为质量，而不是简单追求行为数量。")
add_para(doc, "第三，时间因素呈现出较强的过程性特征。累计 elapsed_time 与正确率的线性相关较弱，但分位组和标准化进程分析显示，不同表现组在游戏早期、中期和后期的行为节奏并不完全一致。这说明总时长不是最有解释力的指标，真正值得关注的是学生在不同阶段的操作密度、操作间隔和关键事件占比。")
add_para(doc, "第四，高级EDA揭示了更细的过程差异。事件转移矩阵从“行为顺序”角度展示了学生操作路径，空间密度图从“交互位置”角度展示了学生注意力和探索范围。这两个视角补充了传统描述统计的不足，使报告能够从结果表现延伸到学习过程本身。")

add_heading(doc, "6.2 实践启示", 2)
add_para(doc, "对于教师或课程设计者而言，本研究的结果说明，教育游戏数据可以作为过程性评价的补充材料。教师不仅可以关注学生最终答对了多少题，也可以观察学生是否在中期题目出现明显困难，是否存在过多重复操作，是否在游戏后半段出现节奏异常。这些信息有助于更早发现学习困难，而不是等到最终成绩出来后再进行补救。")
add_para(doc, "对于游戏设计者而言，题目难度和等级阶段的可视化结果可以帮助检查难度梯度是否平滑。若某些题目长期处于极低正确率，可能需要重新审视题目提示、前置信息或交互设计；若某一阶段整体正确率明显下降，则可以考虑增加过渡引导、提示信息或反馈机制。")
add_para(doc, "对于数据分析过程本身，本研究表明，高级EDA并不等于复杂建模。即使不构建预测模型，也可以通过合理的数据整理、分组比较、时间标准化、转移矩阵和空间密度图，得到具有解释力的发现。这与本课程强调的“提出清楚问题、选择合适方法、用恰当图表表达结果”是一致的。")

add_heading(doc, "6.3 图表展示有效性评价", 2)
add_para(doc, "基础图表（柱状图、箱线图、相关热力图）适合展示分布、排序和变量关系；高级图表（标准化进程折线图、事件转移矩阵、空间密度图）适合展示过程差异和空间特征。整体上，图表组合覆盖了课程要求中的数据分布、变量关系、类别差异、时间变化和空间特征。")
add_para(doc, "从表达效果看，图1至图5主要完成“结果分布”和“题目难度”的说明，适合作为报告前半部分的基础证据；图6至图11逐步转向行为差异和变量关系，帮助解释为什么不同学生会有不同表现；图12至图14则进一步展示时间、路径和空间三个过程维度，使分析不只停留在结果描述，而是能够呈现学生在游戏过程中如何行动。")
add_para(doc, "同时，也需要承认不同图表各有适用边界。雷达图适合做概览，但不适合承载精确数值比较；空间密度图能展示交互集中区域，但如果没有具体场景地图，就不宜对某个坐标点做过度解释；事件转移矩阵能展示路径模式，但仍然是探索性结果，不能直接推出因果关系。")

add_heading(doc, "6.4 不足与改进", 2)
for lim in [
    f"采样代表性有限：行为分析仅使用前{SAMPLE_SIZE//10000}万行日志中的{len(adf):,}个有效会话，可能受到文件排序影响。后续可采用按session_id随机抽样或分层抽样，以增强样本代表性。",
    "elapsed_time 存在长间隔和极端值，不能直接等同于真实连续学习时长。后续可对异常时间间隔进行更精细的截尾、分段或会话重构。",
    "空间坐标只反映屏幕或房间中的交互位置，若不结合具体场景地图，解释仍有局限。后续若能恢复游戏地图或关键区域，可进一步分析学生是否关注到关键线索。",
    "本研究主要是探索性分析，相关关系和路径差异不能直接解释为因果关系。后续可以结合实验设计、更多背景变量或学习内容信息进行验证。",
    "后续还可以采用Markov链指标、聚类分群或轻量级预测模型进一步验证发现，但模型应服务于解释问题，而不是取代可视化分析本身。",
]:
    add_para(doc, lim)

doc.add_page_break()
add_heading(doc, "参考文献", 1)
for ref in [
    "[1] Kaggle. Predict Student Performance from Game Play [EB/OL]. https://www.kaggle.com/competitions/predict-student-performance-from-game-play, 2023.",
    "[2] McKinney W. Data Structures for Statistical Computing in Python [C]. Proceedings of the 9th Python in Science Conference, 2010.",
    "[3] Hunter J D. Matplotlib: A 2D Graphics Environment [J]. Computing in Science & Engineering, 2007.",
    "[4] Waskom M. seaborn: statistical data visualization [J]. Journal of Open Source Software, 2021.",
    "[5] Virtanen P, et al. SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python [J]. Nature Methods, 2020.",
    "[6] Romero C, Ventura S. Educational Data Mining: A Review of the State of the Art [J]. IEEE Transactions on Systems, Man, and Cybernetics, 2010.",
    "[7] Tufte E R. The Visual Display of Quantitative Information [M]. Graphics Press, 2001.",
]:
    add_para(doc, ref, size=9.5, indent=False)

doc.add_page_break()
add_heading(doc, "原创性声明", 1)
add_para(doc, "本人郑重声明：本课程报告是本人基于公开数据集完成的数据分析与可视化课程成果。报告中的数据整理、统计分析、可视化图表和讨论均围绕课程要求展开，引用资料已在参考文献中列出。")
add_para(doc, "本研究所使用的原始数据来源于 Kaggle 公开竞赛平台，仅用于课程学习和学术分析。")
for _ in range(2):
    doc.add_paragraph()
for line in ["学生姓名：海金锁", "学    号：2023110721", "日    期：2026年6月  日"]:
    add_para(doc, line, size=14, align=WD_ALIGN_PARAGRAPH.RIGHT, indent=False)

out_path = ROOT / "数据分析与可视化_图表详析版.docx"
doc.save(out_path)
print("报告已保存:", out_path)
print("图表数量:", len(fig_paths))

报告已保存: D:\课程\数据分析\数据分析与可视化_图表详析版.docx
图表数量: 14


## 7. 输出检查

In [10]:
report_path = ROOT / "数据分析与可视化_图表详析版.docx"
required_figures = [FIG_DIR / f"fig{i:02d}.png" for i in range(1, 12)] + [
    FIG_DIR / "fig12_time_process.png",
    FIG_DIR / "fig13_transition_matrix.png",
    FIG_DIR / "fig14_spatial_density.png",
]
missing = [str(p) for p in required_figures if not p.exists()]
assert not missing, missing
assert report_path.exists() and report_path.stat().st_size > 500_000

with zipfile.ZipFile(report_path) as z:
    media = [n for n in z.namelist() if n.startswith("word/media/")]
    assert len(media) >= 14, len(media)

print("输出检查通过")
print("Notebook:", NB_PATH if "NB_PATH" in globals() else "当前 Notebook")
print("Report:", report_path)
print("Media images in docx:", len(media))

输出检查通过
Notebook: 当前 Notebook
Report: D:\课程\数据分析\数据分析与可视化_图表详析版.docx
Media images in docx: 14
